# Clasificación Multiclase

## Introducción al Problema de Negocio

En el mundo real, muchos problemas de clasificación no son binarios (Sí/No), sino que implican distinguir entre **tres o más categorías**. Esto se conoce como **Clasificación Multiclase**.

### Ejemplos en Business Analytics
*   **Marketing**: Clasificar clientes en segmentos (*VIP, Recurrente, Nuevo, En riesgo*).
*   **Gestión de Riesgos**: Clasificar solicitudes de préstamo por nivel de riesgo (*Bajo, Medio, Alto, Muy Alto*).
*   **Atención al Cliente**: Clasificar incidencias por temática (*Facturación, Técnico, Envíos, Reclamaciones*).

### Objetivo de la Sesión
Trabajaremos con el dataset de **Hotel Bookings** para predecir el **Segmento de Mercado (`market_segment`)** de una reserva. 
Entender el segmento de mercado es vital para que el hotel pueda:
1.  Personalizar ofertas.
2.  Prever cancelaciones (los grupos se comportan distinto a los clientes directos).
3.  Optimizar canales de distribución.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modelos y métricas
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Configuración visual
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Carga y Exploración de Datos

In [ ]:
# Cargamos el dataset
df = pd.read_csv("https://raw.githubusercontent.com/ami232/nebrija-mpan/main/data/hotel_bookings.csv")

# Vista preliminar
df.head()

In [ ]:
# Analizamos la variable objetivo: 'market_segment'
plt.figure(figsize=(10, 5))
sns.countplot(
    x="market_segment", data=df, order=df["market_segment"].value_counts().index
)
plt.title("Distribución de Segmentos de Mercado")
plt.xticks(rotation=45)
plt.show()

print(df["market_segment"].value_counts())

## 2. Preprocesamiento de Datos

In [ ]:
# 1. Eliminaremos la clase 'Undefined' y 'Aviation' por tener muy pocas observaciones.
df = df[~df["market_segment"].isin(["Undefined", "Aviation"])].copy()

# 2. Selección de variables relevantes (ejemplo)
features = [
    "lead_time",
    "total_of_special_requests",
    "required_car_parking_spaces",
    "booking_changes",
    "previous_cancellations",
    "is_repeated_guest",
    "adults",
    "children",
    "babies",
    "customer_type",
    "deposit_type",
    "hotel",
]

target = "market_segment"

# Rellenar nulos si los hay (niños suele tener nulos)
df["children"] = df["children"].fillna(0)

X = df[features]
y = df[target]

# Convertir variables predictoras categóricas con One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# # Si nuestro modelo lo requiere, codificamos la variable objetivo también
# LabelEncoder asigna un número a cada clase. Esto no convierte la variable en ordinal
# le = LabelEncoder()
# y_encoded = le.fit_transform(y)
# print("Clases detectadas:", le.classes_)
# print("Shape final de X:", X.shape)

In [ ]:
print("Clases detectadas:", y.unique())
print("Shape final de X:", X.shape)

### Train-Test Split
Dividimos en conjunto de entrenamiento y prueba.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,  # Estratificar también es importante en multiclase para mantener proporciones
)

# Escalamos para tener coeficientes comparables y estabilidad numérica en la Regresión Logística
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Teoría: Estrategias para Clasificación Multiclase

Cuando tenemos más de dos clases ($K > 2$), existen principalmente tres enfoques para adaptar los algoritmos:

### 1. Modelos One-vs-Rest (OvR) o "Una contra las demás"
*  Se entrenan $K$ clasificadores binarios independientes.
*  El clasificador $i$ aprende a distinguir la clase $i$ (clase positiva) de "todas las demás" (clase negativa).
*  Para predecir, se ejecutan los $K$ modelos y se asigna la clase del que tenga mayor predicción de probabilidad.

### 2. Modelos One-vs-One (OvO) o "Uno contra Uno"
*   Se entrena un clasificador para cada **par** de clases posible.
*   Si tenemos $K$ clases, necesitamos entrenar $\frac{K(K-1)}{2}$ modelos. (Con 4 clases = 6 modelos).
    *   Ej: *Modelo A vs B*, *Modelo A vs C*, *Modelo B vs C*...
*   Para predecir, cada modelo vota por una de las dos clases que compara. Gana la clase que reciba más votos en total.
*   Es útil para algoritmos que escalan mal con el número de datos (como SVM), ya que cada sub-modelo solo entrena con una pequeña parte de los datos (solo las filas de esas 2 clases).

### 3. Regresión Logística Multinomial / Softmax
*  Para cada clase $k$, el modelo calcula un modelo lineal $z_k = w_k^T x + b_k$. 
En lugar de la función Sigmoide (que da un valor 0-1), usa la función **Softmax**. Esta función convierte los scores del modelo en una distribución de probabilidad sobre las $K$ clases.
    *   $$P(y=k|x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$
*   Al contrario que en modelos OvR/OvO, todos los modelos lineales se entrenan simultáneamente y comparten la misma función objetivo (Cross-Entropy Loss). 
*   Las probabilidades de todas las clases suman 1.

### 4. Árboles de decisión
*   Los modelos basados en árboles de decisión adminten multiclase por diseño sin modifiacaciones adicionales. La predicción se realiza evaluando las reglas del árbol hasta llegar a una hoja.
*  Se predice la clase mayoritaria en esa hoja (o la distribución de clases en esa hoja si se desea obtener probabilidades).

## Ejemplo de One-vs-Rest (OvR) con Regresión Logística

In [ ]:
# Crear un clasificador One-vs-Rest con Regresión Logística como estimador base
model_ovr = OneVsRestClassifier(LogisticRegression(max_iter=1000))

# Entrenar el modelo
model_ovr.fit(X_train_scaled, y_train)

# Realizar predicciones
y_pred_ovr = model_ovr.predict(X_test_scaled)

# Evaluar
print("=" * 60)
print("ONE-VS-REST (OvR) CON REGRESIÓN LOGÍSTICA")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred_ovr):.4f}")

print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_ovr, target_names=model_ovr.classes_))

# Visualizar probabilidades OvR
print("\n" + "=" * 60)
print("PROBABILIDADES ASIGNADAS (One-vs-Rest)")
print("=" * 60)

probs_ovr = model_ovr.predict_proba(X_test_scaled[:5])
df_probs_ovr = pd.DataFrame(probs_ovr, columns=model_ovr.classes_)
df_probs_ovr["Predicción OvR"] = model_ovr.predict(X_test_scaled[:5])
print(
    "Hay warnings porque la clase 'Complementary' no tiene predicciones en el conjunto de test, por lo que la Precision no está definida para esa clase."
)

print("\n")
display(df_probs_ovr.round(3))

# Matriz de Confusión para OvR
print("\n" + "=" * 60)
print("MATRIZ DE CONFUSIÓN - ONE-VS-REST")
print("=" * 60)

plt.figure(figsize=(12, 10))
cm_ovr = confusion_matrix(y_test, y_pred_ovr)
disp_ovr = ConfusionMatrixDisplay(
    confusion_matrix=cm_ovr, display_labels=model_ovr.classes_
)
disp_ovr.plot(cmap="Greens", xticks_rotation="vertical")
plt.title("Matriz de Confusión - One-vs-Rest (OvR)")
plt.grid(False)
plt.show()

## Ejemplo de One-vs-One (OvO) con Regresión Logística

In [ ]:
# Crear un clasificador One-vs-One con Regresión Logística como estimador base
model_ovo = OneVsOneClassifier(LogisticRegression(max_iter=1000))

# Entrenar el modelo
model_ovo.fit(X_train_scaled, y_train)

# Realizar predicciones
y_pred_ovo = model_ovo.predict(X_test_scaled)

# Evaluar
print("=" * 60)
print("ONE-VS-ONE (OvO) CON REGRESIÓN LOGÍSTICA")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred_ovo):.4f}")

print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_ovo, target_names=model_ovo.classes_))

# Matriz de Confusión para OvO
print("\n" + "=" * 60)
print("MATRIZ DE CONFUSIÓN - ONE-VS-ONE")
print("=" * 60)

plt.figure(figsize=(12, 10))
cm_ovo = confusion_matrix(y_test, y_pred_ovo)
disp_ovo = ConfusionMatrixDisplay(
    confusion_matrix=cm_ovo, display_labels=model_ovo.classes_
)
disp_ovo.plot(cmap="Purples", xticks_rotation="vertical")
plt.title("Matriz de Confusión - One-vs-One (OvO)")
plt.grid(False)
plt.show()

## OvO vs OvR
Fíjate en las matrices de confusión arriba.
*   **OvR** es más rápido (menos modelos), pero puede sufrir si las clases están muy desbalanceadas (ej: clase "Complementary" no se predice nunca en OvR).
*   **OvO** es más lento de entrenar con muchos datos (muchos modelos), pero suele ser más robusto porque cada sub-modelo solo trabaja con clases bien definidas y solo con las filas de las clases en cuestión.

En la práctica con Scikit-Learn:
*   Si usas `LogisticRegression`, por defecto usa OvR, especialmente si tienes muchas clases
*   Modelos basados en árboles no necesitan OvO ni OvR

## Regresión Logística Multinomial

La Regresión Logística estándar es binaria. Para usarla con múltiples clases, usamos el enfoque **Multinomial (Softmax)**.

- **Scikit-learn** hace esto automáticamente si detecta múltiples clases, o podemos especificar `multi_class='multinomial'`.

In [ ]:
# Entrenamos el modelo
# max_iter aumentado porque multinomial puede tardar en converger
model_log = LogisticRegression(max_iter=1000)
model_log.fit(X_train_scaled, y_train)

### Evaluación del Modelo

In [ ]:
y_pred_log = model_log.predict(X_test_scaled)

# Accuracy general
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")

# Reporte de clasificación por clase
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_log, target_names=model_log.classes_))

El modelo **no predice directamente la clase**.

1.  Internamente, el modelo calcula un "score" para cada una de las clases posibles.
2.  Como es un modelo probabilístico, asigna una **probabilidad a cada clase** (que suman 100%).
3.  Finalmente, **elige la clase con la probabilidad más alta**.

Podemos ver estas probabilidades con `.predict_proba()`:

In [ ]:
# Veamos las probabilidades para las primeras 5 predicciones
probs = model_log.predict_proba(X_test_scaled[:5])

# Creamos un DataFrame para verlo bonito
df_probs = pd.DataFrame(probs, columns=model_log.classes_)
df_probs["Predicción"] = model_log.predict(X_test_scaled[:5])

# Mostramos
print("Probabilidades asignadas por el modelo:")
display(df_probs.round(3))

In [ ]:
# Matriz de Confusión Visual
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred_log)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model_log.classes_)
disp.plot(cmap="Blues", xticks_rotation="vertical")
plt.title("Matriz de Confusión - Regresión Logística")
plt.grid(False)
plt.show()

**Análisis**: Observa la diagonal principal (aciertos) y dónde se confunde el modelo (fuera de la diagonal). ¿Qué segmentos se confunden más entre sí?

## Random Forest

Los árboles de decisión y random forest manejan naturalmente la clasificación multiclase sin necesidad de trucos One-vs-All.

In [ ]:
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(
    X_train, y_train
)  # Usamos X_train sin escalar (aunque con escalar funcionaría igual)

y_pred_rf = model_rf.predict(X_test)

print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nReporte de Clasificación Random Forest:")
print(classification_report(y_test, y_pred_rf, target_names=model_log.classes_))

### AUC-ROC en Multiclase

El área bajo la curva ROC (AUC) está definida originalmente para clasificación binaria. Para usarla en multiclase, hacemos lo mismo que con los modelos: usar estrategias **OvR (One-vs-Rest)** o **OvO (One-vs-One)**.

1.  **Macro-Average AUC**: Hacemos el promedio simple del AUC de cada clase (si queremos tratar todas las clases por igual).
2.  **Weighted-Average AUC**: Promedio ponderado por el número de muestras (si queremos dar más peso a clases con más observaciones).

In [ ]:
for model in [model_ovr, model_log]:
    y_prob = model.predict_proba(X_test_scaled)
    print(model)
    auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="macro")
    print(f"AUC-ROC Multiclase (Macro OvR): {auc:.4f}")
    auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")
    print(f"AUC-ROC Multiclase (Weighted OvR): {auc:.4f}")

## Clases desbalanceadas

En problemas multiclase, es muy común tener categorías con muy pocas observaciones. Esto causa varios problemas:
1.  **Soporte Insuficiente**: El modelo no puede aprender patrones si solo tiene 3 o 4 ejemplos de entrenamiento.
2.  **Métricas indefinedas o engañosas**: El Accuracy global puede parecer alto porque acierta la clase mayoritaria, pero falla completamente en las minoritarias. También es posible que no se prediga 
3.  **Train/Test Split**: Al dividir los datos, es posible que **ninguna** observación de la clase pequeña caiga en el conjunto de Test (o peor, ninguna en el de Train), lo que hace imposible entrenar o evaluar esa clase.

**Ejemplo Práctico en nuestro Dataset**:
La clase `Undefined` o `Aviation` tienen muy pocos registros. Si intentáramos evaluar un modelo incluyendo clases raras sin suficiente representación, obtendríamos errores o métricas de 0.00 en esas clases.
*   *Solución común*: Agrupar categorías pequeñas en una etiqueta "Otros" o eliminarlas si no son críticas para el negocio.
*   *Técnica técnica*: Usar `stratify=y` en el split asegura (si hay suficientes datos) que las proporciones se mantengan, evitando que una clase desaparezca del todo en el test set.

--- 
## Ejercicios para Clase

### Ejercicio 1: Interpretación de Errores
Mira la matriz de confusión del Random Forest. 
1. Genera la gráfica de la matriz de confusión para el Random Forest.
2. ¿Cuál es el segmento de mercado que mejor predecimos? ¿Y el peor?
3. ¿Existe alguna confusión lógica? (Ej: ¿Confundimos 'Offline TA/TO' con 'Groups'?)

In [ ]:
# TU CÓDIGO AQUÍ

### Ejercicio 2: Predicción de Tipo de Cliente

Ahora intenta predecir la variable `customer_type` en lugar de `market_segment`. 
1. Define tu nueva `y` con la columna `customer_type`.
2. Divide en train y test (recuerda usar `stratify`).
3. Entrena un modelo (LogReg o RandomForest) y evalúa los resultados con un `classification_report`.
4. ¿Qué clase está muy desbalanceada y cómo afecta eso a las métricas (precision/recall) de esa clase?

In [ ]:
# TU CÓDIGO AQUÍ